# 04. Late binding

**Tier:** Intermediate. **Why you'd use this:** You build a list of handler functions in a loop, each meant to remember its own index. Closures look up the variable when they run, so every handler sees the final index.

**After this lesson you can:** bind the loop value as a default; explain why the closure sees the late value.
About 25 minutes. **Uses:** [Scope](../05-functions/03-scope-lesson.ipynb)


## Quick review

1. When should you use `is` instead of `==`?
2. Why are equal list literals still different objects?

<details><summary>Answers</summary>

For `None` and other singletons. Each literal builds a new list.
</details>


## The idea

A nested function that reads a name from the enclosing scope looks up that name when it is called, not when it was defined. In a loop, that name is the one cell that keeps changing. By call time it holds the last value. Capture the current value as a default argument, which is evaluated at `def` time for that iteration.

<details><summary>Explain more</summary>

`def handler(n=n):` stores today's `n` in `__defaults__`. The body uses that parameter, not the loop cell.
</details>


## See it work

Build handlers that each remember their own number.


In [1]:
def make_handlers(count):
    handlers = []
    for n in range(count):
        # 1. Default binds the current n at def time.
        def handler(n=n):
            return n

        handlers.append(handler)
    return handlers


for handle in make_handlers(3):
    print(handle())

0
1
2


## Check yourself

Answer these before you run the cell.

1. What do all three broken handlers return?
2. Why does `n=n` fix it?


In [2]:
handlers = []
for n in range(3):

    def handler():
        return n

    handlers.append(handler)

print([handle() for handle in handlers])

[2, 2, 2]


<details><summary>Answers</summary>

All return `2`. The default `n=n` stores each iteration's value on that function when it is defined.
</details>


## Why it works this way

Closures close over cells, not snapshots of values. The loop variable is one cell. Each nested `def` without a default keeps a reference to that cell. When the handler finally runs, it reads whatever the cell holds now, which after the loop is the last value. Deep dive: [Scopes and closures](../../internals/06-scopes/01-closures-lesson.ipynb).

Default arguments are evaluated at function definition time and stored on that function object. Writing `def handler(n=n)` during iteration creates a new function whose `__defaults__` holds the current integer. The body then reads the parameter, which is ordinary local lookup, not the loop cell.

Cost is one small function object per handler and a one-element defaults tuple. The design tension is late binding for closures (useful when you want the latest value) versus the loop pitfall (when you wanted a snapshot). Defaults are the standard snapshot. Another fix is to push the value through an outer factory call `make(n)` that returns the handler, because the parameter `n` in `make` is a new local each call.

Evidence: three broken handlers all return the final index, and three fixed handlers return `0, 1, 2`. Looking at `__defaults__` shows the captured numbers. If you inspect a broken handler's closure cell after the loop, every function points at the same cell with the final value. That shared cell is the whole mechanism. Late binding is useful when you want the latest value on purpose. In a loop of callbacks, you almost never want that. Snapshot with a default or a factory.


In [3]:
broken = []
fixed = []
for n in range(3):

    def handler():
        return n

    broken.append(handler)

    def handler(n=n):
        return n

    fixed.append(handler)

print([handle() for handle in broken])
print([handle() for handle in fixed])
print([handle.__defaults__ for handle in fixed])

[2, 2, 2]
[0, 1, 2]
[(0,), (1,), (2,)]


## When to use it

Use a default capture (or a factory) when a closure must remember the loop value. Use a plain closure when you intentionally want the latest enclosing value. Don't build handler lists in a loop without one of those captures. Event callbacks and button handlers are the usual place this bug shows up.


## Common mistake

Appending `lambda: n` in a loop and expecting each lambda to keep its own `n`.


In [4]:
handlers = []
for n in range(3):
    handlers.append(lambda: n)

print([handle() for handle in handlers])

[2, 2, 2]


Bind with a default: `lambda n=n: n`, or use a small factory function.


In [5]:
handlers = []
for n in range(3):
    handlers.append(lambda n=n: n)

print([handle() for handle in handlers])

[0, 1, 2]


## Try it

Rewrite `make_handlers` with an inner factory `def make(n): ...` instead of a default on the handler. Which name is the cell that no longer changes under the handler?


## Recap

- Closures look up enclosing names at call time.
- A loop variable is one cell shared by every iteration's closure.
- Capture with a default or a factory.

**Next:** [practice](04-late-binding-practice.ipynb)
